# Bone Age Estimation from Pediatric Hand X-rays

**A Deep Learning Regression Approach using Transfer Learning with ResNet50**

**Author:** Keerthi P R  
**Institution:** St Joseph's University, Bengaluru — MSc Big Data Analytics  
**Registration No.:** 252BDA04  
**Dataset:** RSNA Pediatric Bone Age Challenge (Kaggle)

---

## Problem Statement

Automate the estimation of pediatric bone age (in months) from hand X-ray images. Bone age assessment is a critical clinical procedure for diagnosing growth disorders. Manual assessment takes 15–20 minutes per case and shows significant inter-observer variability. This project frames the problem as a **regression task** rather than classification for clinical precision.

## Approach

- Transfer learning with **ResNet50** (frozen, ImageNet pre-trained)
- Custom regression head with dropout regularization
- MAE loss + Adam optimizer
- Comprehensive statistical evaluation (Bland-Altman, age-stratified analysis)

## Section 1: Environment Setup

In [ ]:
# Verify GPU is available
import tensorflow as tf
print('GPU available:', tf.config.list_physical_devices('GPU'))
print('TensorFlow version:', tf.__version__)

In [ ]:
# Upload Kaggle API credentials (kaggle.json file)
from google.colab import files
files.upload()  # Upload your kaggle.json here

In [ ]:
# Set up Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download and unzip the RSNA Bone Age dataset
!kaggle datasets download -d kmader/rsna-bone-age
!unzip -q rsna-bone-age.zip -d bone_age_data

## Section 2: Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random

# Load labels
df = pd.read_csv('bone_age_data/boneage-training-dataset.csv')
print(f'Total training images: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Descriptive statistics
print('Bone Age Statistics (in months):')
print(df['boneage'].describe())
print(f"\nSex breakdown:")
print(df['male'].value_counts())
print(f"\nAge range: {df['boneage'].min()} to {df['boneage'].max()} months")
print(f"That's roughly {df['boneage'].min()/12:.1f} to {df['boneage'].max()/12:.1f} years old")

In [ ]:
# Age distribution visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['boneage'], bins=40, edgecolor='black', color='steelblue')
axes[0].axvline(df['boneage'].mean(), color='red', linestyle='--', label=f"Mean: {df['boneage'].mean():.1f}")
axes[0].axvline(df['boneage'].median(), color='green', linestyle='--', label=f"Median: {df['boneage'].median():.1f}")
axes[0].set_xlabel('Bone Age (months)')
axes[0].set_ylabel('Number of Children')
axes[0].set_title('Distribution of Bone Ages')
axes[0].legend()

axes[1].hist(df[df['male']==True]['boneage'], bins=40, alpha=0.6, label='Boys', color='blue', edgecolor='black')
axes[1].hist(df[df['male']==False]['boneage'], bins=40, alpha=0.6, label='Girls', color='pink', edgecolor='black')
axes[1].set_xlabel('Bone Age (months)')
axes[1].set_ylabel('Number of Children')
axes[1].set_title('Age Distribution by Sex')
axes[1].legend()

plt.tight_layout()
plt.savefig('age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize sample X-rays
sample_ids = random.sample(list(df['id']), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, img_id in zip(axes.flatten(), sample_ids):
    img_path = f'bone_age_data/boneage-training-dataset/boneage-training-dataset/{img_id}.png'
    img = Image.open(img_path)
    row = df[df['id'] == img_id].iloc[0]
    age = row['boneage']
    sex = 'Boy' if row['male'] else 'Girl'
    ax.imshow(img, cmap='gray')
    ax.set_title(f'ID: {img_id} | {sex} | Age: {age} months ({age/12:.1f} yrs)')
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_xrays.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 3: Data Preprocessing & Pipeline

In [ ]:
from sklearn.model_selection import train_test_split

# 80/10/10 split
train_val_df, test_df = train_test_split(df, test_size=0.10, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.111, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Training:   {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)')
print(f'Validation: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test:       {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)')

In [ ]:
# Prepare filename column and convert boneage to float
IMG_SIZE = 224
BATCH_SIZE = 32
IMG_DIR = 'bone_age_data/boneage-training-dataset/boneage-training-dataset/'

for d in [train_df, val_df, test_df]:
    d['filename'] = d['id'].astype(str) + '.png'
    d['boneage'] = d['boneage'].astype('float32')

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation/test generators - NO augmentation
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# Connect generators
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df, directory=IMG_DIR,
    x_col='filename', y_col='boneage',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='raw',
    color_mode='rgb', shuffle=True, seed=42
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df, directory=IMG_DIR,
    x_col='filename', y_col='boneage',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='raw',
    color_mode='rgb', shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df, directory=IMG_DIR,
    x_col='filename', y_col='boneage',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='raw',
    color_mode='rgb', shuffle=False
)

## Section 4: Model Architecture (Transfer Learning with ResNet50)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Load ResNet50 without classification head
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze base model
base_model.trainable = False

# Build custom regression head
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(1, activation='linear')(x)  # Regression output

model = Model(inputs, outputs)

# Compile with MAE loss for regression
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mean_absolute_error',
    metrics=['mae', 'mse']
)

model.summary()

## Section 5: Training

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_bone_age_model.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

steps_per_epoch = len(train_df) // BATCH_SIZE
validation_steps = len(val_df) // BATCH_SIZE

history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# IMMEDIATELY save model + history to Google Drive
from google.colab import drive
drive.mount('/content/drive')

!cp best_bone_age_model.keras /content/drive/MyDrive/

import pickle
with open('/content/drive/MyDrive/training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

print('Model and history saved to Google Drive!')

## Section 6: Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Training MAE', color='steelblue', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation MAE', color='darkorange', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean Absolute Error (months)')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mse'], label='Training MSE', color='steelblue', linewidth=2)
axes[1].plot(history.history['val_mse'], label='Validation MSE', color='darkorange', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Squared Error')
axes[1].set_title('Training vs Validation MSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest Validation MAE: {min(history.history['val_loss']):.2f} months")

## Section 7: Test Set Evaluation

In [ ]:
from tensorflow.keras.models import load_model

# Load the best saved model
best_model = load_model('best_bone_age_model.keras')

# Evaluate on test set
test_results = best_model.evaluate(test_generator, verbose=1)
print(f'\nTest MAE: {test_results[1]:.2f} months')
print(f'Test MSE: {test_results[2]:.2f}')
print(f'Test RMSE: {test_results[2]**0.5:.2f} months')

In [ ]:
# Generate predictions on test set
test_predictions = best_model.predict(test_generator, verbose=1).flatten()
test_actuals = test_df['boneage'].values[:len(test_predictions)]

# Compute detailed metrics
mae = np.mean(np.abs(test_predictions - test_actuals))
mse = np.mean((test_predictions - test_actuals) ** 2)
rmse = np.sqrt(mse)

ss_res = np.sum((test_actuals - test_predictions) ** 2)
ss_tot = np.sum((test_actuals - np.mean(test_actuals)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

correlation = np.corrcoef(test_predictions, test_actuals)[0, 1]
residuals = test_predictions - test_actuals
mean_residual = np.mean(residuals)
std_residual = np.std(residuals)

print('='*50)
print('COMPREHENSIVE TEST METRICS')
print('='*50)
print(f'MAE:              {mae:.2f} months')
print(f'RMSE:             {rmse:.2f} months')
print(f'R² Score:         {r_squared:.4f}')
print(f'Pearson r:        {correlation:.4f}')
print(f'Mean Residual:    {mean_residual:.2f} months')
print(f'Std of Residuals: {std_residual:.2f} months')
print('='*50)

## Section 8: Predictions vs Actuals + Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Predicted vs Actual
axes[0].scatter(test_actuals, test_predictions, alpha=0.4, color='steelblue', s=15)
axes[0].plot([0, 250], [0, 250], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Bone Age (months)', fontsize=12)
axes[0].set_ylabel('Predicted Bone Age (months)', fontsize=12)
axes[0].set_title(f'Predicted vs Actual (R² = {r_squared:.3f})', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
axes[1].scatter(test_actuals, residuals, alpha=0.4, color='darkorange', s=15)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[1].axhline(y=mean_residual, color='green', linestyle=':', linewidth=2, label=f'Mean: {mean_residual:.2f}')
axes[1].set_xlabel('Actual Bone Age (months)', fontsize=12)
axes[1].set_ylabel('Residual (Predicted − Actual)', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predictions_vs_actuals.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 9: Bland-Altman Analysis (Clinical Method Agreement)

In [ ]:
# Bland-Altman analysis
mean_measurements = (test_predictions + test_actuals) / 2
difference = test_predictions - test_actuals

mean_diff = np.mean(difference)
std_diff = np.std(difference)
lower_loa = mean_diff - 1.96 * std_diff
upper_loa = mean_diff + 1.96 * std_diff
within_loa = np.sum((difference >= lower_loa) & (difference <= upper_loa)) / len(difference) * 100

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(mean_measurements, difference, alpha=0.4, color='steelblue', s=15)
ax.axhline(y=mean_diff, color='red', linestyle='-', linewidth=2, label=f'Mean Difference: {mean_diff:.2f}')
ax.axhline(y=upper_loa, color='green', linestyle='--', linewidth=2, label=f'Upper 95% LoA: {upper_loa:.2f}')
ax.axhline(y=lower_loa, color='green', linestyle='--', linewidth=2, label=f'Lower 95% LoA: {lower_loa:.2f}')
ax.axhline(y=0, color='black', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xlabel('Mean of Predicted and Actual Age (months)', fontsize=12)
ax.set_ylabel('Predicted − Actual (months)', fontsize=12)
ax.set_title('Bland-Altman Plot: Model Predictions vs Radiologist Labels', fontsize=13)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bland_altman.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBland-Altman:')
print(f'  Mean Difference:  {mean_diff:.2f} months')
print(f'  95% LoA:          [{lower_loa:.2f}, {upper_loa:.2f}] months')
print(f'  Coverage:         {within_loa:.1f}%')

## Section 10: Age-Stratified Error Analysis

In [ ]:
age_bins = [0, 24, 60, 108, 156, 204, 228]
age_labels = ['0-2y', '2-5y', '5-9y', '9-13y', '13-17y', '17-19y']

test_results_df = pd.DataFrame({
    'actual': test_actuals,
    'predicted': test_predictions,
    'abs_error': np.abs(test_predictions - test_actuals),
    'residual': test_predictions - test_actuals
})
test_results_df['age_group'] = pd.cut(test_results_df['actual'], bins=age_bins, labels=age_labels)

group_stats = test_results_df.groupby('age_group', observed=True).agg(
    count=('actual', 'count'),
    mae=('abs_error', 'mean'),
    rmse=('residual', lambda x: np.sqrt(np.mean(x**2))),
    mean_residual=('residual', 'mean')
).round(2)

print('Error Metrics by Age Group:')
print(group_stats)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(group_stats.index.astype(str), group_stats['mae'], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('MAE (months)')
axes[0].set_title('Prediction Error by Age Group')
axes[0].axhline(y=mae, color='red', linestyle='--', label=f'Overall MAE: {mae:.1f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(group_stats.index.astype(str), group_stats['count'], color='darkorange', edgecolor='black')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Number of Test Samples')
axes[1].set_title('Test Set Distribution by Age Group')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('age_stratified_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 11: Download All Artifacts (IMMEDIATELY)

In [ ]:
# Download everything to laptop - do this IMMEDIATELY after training completes
from google.colab import files

files.download('best_bone_age_model.keras')
files.download('training_curves.png')
files.download('predictions_vs_actuals.png')
files.download('bland_altman.png')
files.download('age_stratified_analysis.png')
files.download('age_distribution.png')
files.download('sample_xrays.png')